<a href="https://colab.research.google.com/github/andreelzs/Linguagens-de-programacao/blob/main/Explorando_Dados_na_Web_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pandas -q

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# NÍVEL 1

In [3]:
# 1.1
resp = requests.get('https://jsonplaceholder.typicode.com/posts', timeout=10)
dados = resp.json()
print(f'Status: {resp.status_code}')
print(f'Posts: {len(dados)}')
print(dados[0])

Status: 200
Posts: 100
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [4]:
# 1.2
params = {'userId': 2, '_limit': 5}
resp = requests.get('https://jsonplaceholder.typicode.com/posts', params=params, timeout=10)
print(f'URL: {resp.url}')
print(f'Status: {resp.status_code}')
for post in resp.json():
    print(post['title'])

URL: https://jsonplaceholder.typicode.com/posts?userId=2&_limit=5
Status: 200
et ea vero quia laudantium autem
in quibusdam tempore odit est dolorem
dolorum ut in voluptas mollitia et saepe quo animi
voluptatem eligendi optio
eveniet quod temporibus


In [5]:
# 1.3
headers = {'User-Agent': 'MeuProjeto/1.0'}
resp = requests.get('https://jsonplaceholder.typicode.com/posts', params={'userId': 1, '_limit': 3}, headers=headers, timeout=10)
print(f'Status: {resp.status_code}')
print(f'Headers enviados: {headers}')
print(resp.json())

Status: 200
Headers enviados: {'User-Agent': 'MeuProjeto/1.0'}
[{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}, {'userId': 1, 'id': 3, 'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'}]


In [6]:
# 1.4
resp = requests.get('https://jsonplaceholder.typicode.com/users/1', timeout=10)
print(f'Status: {resp.status_code}')
print(f'URL final: {resp.url}')

Status: 200
URL final: https://jsonplaceholder.typicode.com/users/1


# NÍVEL 2

In [7]:
# 2.1
ceps = ['01310100', '20040020', '30150371']
resultados = []

for cep in ceps:
    try:
        resp = requests.get(f'https://viacep.com.br/ws/{cep}/json/', timeout=10)
        dados = resp.json()
        if 'erro' not in dados:
            resultados.append({'CEP': cep, 'Cidade': dados['localidade'], 'UF': dados['uf']})
    except Exception as e:
        print(f'Erro {cep}: {e}')

df = pd.DataFrame(resultados)
print(df)

        CEP          Cidade  UF
0  01310100       São Paulo  SP
1  20040020  Rio de Janeiro  RJ


In [8]:
# 2.2
def download_seguro(url):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        return resp.content
    except requests.exceptions.HTTPError as e:
        print(f'Erro HTTP {e.response.status_code}')
        return None
    except Exception as e:
        print(f'Erro: {e}')
        return None

resultado = download_seguro('https://jsonplaceholder.typicode.com/posts/1')
print('OK' if resultado else 'Falhou')

OK


In [9]:
# 2.3
resp = requests.get('https://picsum.photos/400/400', timeout=10)
with open('imagem.jpg', 'wb') as f:
    f.write(resp.content)
print(f'Imagem salva - {len(resp.content)} bytes')

Imagem salva - 31172 bytes


# NÍVEL 3

In [10]:
# 3.1
resp = requests.get('https://books.toscrape.com/robots.txt', timeout=10)
print(resp.text[:500])

<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



In [11]:
# 3.2
resp = requests.get('https://books.toscrape.com/', timeout=10)
soup = BeautifulSoup(resp.content, 'html.parser')

livros = soup.select('.product_pod')[:5]
for livro in livros:
    titulo = livro.select_one('h3 a')['title']
    preco = livro.select_one('.price_color').get_text()
    print(f'{titulo} - {preco}')

A Light in the Attic - £51.77
Tipping the Velvet - £53.74
Soumission - £50.10
Sharp Objects - £47.82
Sapiens: A Brief History of Humankind - £54.23


In [12]:
# 3.3
resp = requests.get('https://books.toscrape.com/', timeout=10)
soup = BeautifulSoup(resp.content, 'html.parser')

livros_data = []
for livro in soup.select('.product_pod'):
    livros_data.append({
        'titulo': livro.select_one('h3 a')['title'],
        'preco': livro.select_one('.price_color').get_text()
    })

df = pd.DataFrame(livros_data)
df.to_csv('livros.csv', index=False)
print(df.head())

                                  titulo   preco
0                   A Light in the Attic  £51.77
1                     Tipping the Velvet  £53.74
2                             Soumission  £50.10
3                          Sharp Objects  £47.82
4  Sapiens: A Brief History of Humankind  £54.23


In [14]:
import requests
import pandas as pd
from io import StringIO

# 3.4
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

try:
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    tabelas = pd.read_html(StringIO(resp.text)) # Wrap resp.text in StringIO
    df = tabelas[0]
    print(df.head())
except requests.exceptions.RequestException as e:
    print(f"Erro ao acessar a URL: {e}")
except ValueError as e:
    print(f"Erro ao analisar o HTML (nenhuma tabela encontrada ou problema de formatação): {e}")

  Country or territory  Population (1 July 2022)  Population (1 July 2023)  \
0                  NaN                       NaN                       NaN   
1                World              8.021407e+09              8.091735e+09   
2                India              1.425423e+09              1.438070e+09   
3             China[a]              1.425180e+09              1.422585e+09   
4        United States              3.415340e+08              3.434773e+08   

  Change (%) UN continental region[1] UN statistical subregion[1]  
0        NaN                      NaN                         NaN  
1     +0.88%                        –                           –  
2     +0.89%                     Asia               Southern Asia  
3     −0.18%                     Asia                Eastern Asia  
4     +0.57%                 Americas            Northern America  


/tmp/ipykernel_1970/1462858085.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(resp.text)
